# Lab 7 — End-to-end Foundry IQ RAG evaluation

This lab evaluates the live Foundry IQ knowledge-base `retrieve` operation in extractive-output mode. IQ plans and executes retrieval; the project model then turns only that retrieved evidence into independently cited factual claims.

Keeping retrieval and answer generation separate makes missing evidence, incomplete answers, bad citations, latency, and cost independently measurable.

## What is measured

- **Retrieval recall@k:** required procedure, authorization, and crew evidence found among actual IQ references.
- **Citation coverage and validity:** every generated factual claim has one or more returned `[ref_id:#]` citations, and every citation resolves.
- **Groundedness and response completeness:** Foundry evaluators compare the answer with the exact retrieved evidence and the versioned expected answer.
- **Repeatability:** every benchmark case runs several times; release gates apply to every run and the case summary reports worst-case quality.
- **Latency and cost:** IQ retrieval and answer generation are timed separately; model tokens, agentic-retrieval tokens, and Search requests stay visible.

Current APIs: [Retrieve from a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve), [Foundry IQ guidance](https://learn.microsoft.com/azure/foundry/agents/concepts/foundry-iq-faq), and [RAG evaluators](https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/rag-evaluators).

In [ ]:
import json
import math
import os
import re
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path
from statistics import median
from urllib.parse import quote

from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    helper = candidate / 'labs' / 'observability-and-evaluation' / 'rag_eval_utils.py'
    if helper.exists():
        repo_root = candidate
        sys.path.insert(0, str(helper.parent))
        break
else:
    raise FileNotFoundError('rag_eval_utils.py not found')

from rag_eval_utils import (
    document_evidence_keys,
    estimate_cost_usd,
    extract_iq_extracted_documents,
    fetch_iq_reference_documents,
    format_iq_context,
    iq_activity_metrics,
    load_cases,
    price_rates_from_env,
    primitive,
    render_cited_answer,
    response_token_usage,
    retrieval_recall,
    validate_cited_answer,
)

load_dotenv(repo_root / '.env')
project_endpoint = os.getenv('FOUNDRY_PROJECT_ENDPOINT') or os.getenv('AZURE_AI_PROJECT_ENDPOINT')
model_deployment = os.getenv('FOUNDRY_MODEL') or os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')
search_endpoint = os.getenv('AZURE_SEARCH_ENDPOINT', '').rstrip('/')
knowledge_base = os.getenv('FOUNDRY_IQ_KNOWLEDGE_BASE')
api_version = os.getenv('FOUNDRY_IQ_API_VERSION', '2026-05-01-preview')
iq_output_mode = os.getenv('FOUNDRY_IQ_OUTPUT_MODE', 'extractiveData')
repeat_count = int(os.getenv('RAG_EVAL_REPEATS', '3'))
judge_repeat_count = int(os.getenv('RAG_EVAL_JUDGE_REPEATS', '3'))
evaluator_model = os.getenv('RAG_EVALUATOR_MODEL') or model_deployment
if not 1 <= repeat_count <= 10:
    raise ValueError('RAG_EVAL_REPEATS must be between 1 and 10')
if not 3 <= judge_repeat_count <= 9 or judge_repeat_count % 2 == 0:
    raise ValueError('RAG_EVAL_JUDGE_REPEATS must be an odd number between 3 and 9')
raw_namespace = (
    os.getenv('WORKSHOP_RESOURCE_NAMESPACE')
    or os.getenv('WORKSHOP_TEAM_ID')
    or os.getenv('WORKSHOP_PARTICIPANT_ID')
    or ''
)
resource_namespace = re.sub(r'[^a-z0-9-]+', '-', raw_namespace.lower()).strip('-')[:32]
required = (project_endpoint, model_deployment, search_endpoint, knowledge_base, resource_namespace)
if not all(required):
    raise ValueError('Missing Foundry IQ, Search, model, or WORKSHOP_RESOURCE_NAMESPACE configuration')
print({'namespace': resource_namespace, 'knowledge_base': knowledge_base, 'api_version': api_version, 'output_mode': iq_output_mode, 'response_repeats': repeat_count, 'judge_repeats': judge_repeat_count, 'generator_model': model_deployment, 'evaluator_model': evaluator_model})

In [ ]:
dataset_path = repo_root / 'labs' / 'observability-and-evaluation' / 'data' / 'rag-evaluation-cases-v2.json'
dataset, iq_cases = load_cases(dataset_path, 'foundry_iq')
assert dataset['dataset_id'] == 'synthetic-grid-rag-e2e-v2'
assert len(iq_cases) >= 2
assert all(set(case['source_filters']) == {'procedure', 'authorization', 'crew'} for case in iq_cases)
print({'dataset': dataset['dataset_id'], 'cases': [row['case_id'] for row in iq_cases]})

## Configure source-aware retrieval

Each benchmark row declares exact, filterable identifiers already present in the request. The lab passes those identifiers as `filterAddOn` controls. The per-source candidate cap follows the preview API's 50–200 contract; the top-level final-document cap remains 12.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import InteractiveBrowserCredential
from azure.search.documents import SearchClient

credential = InteractiveBrowserCredential()
search_token = credential.get_token('https://search.azure.com/.default').token
source_by_role = {
    'procedure': os.getenv('FOUNDRY_IQ_PROCEDURES_SOURCE'),
    'authorization': os.getenv('FOUNDRY_IQ_RAAMOPDRACHTEN_SOURCE'),
    'crew': os.getenv('FOUNDRY_IQ_CREW_SOURCE'),
}
source_to_index = {
    source_by_role['procedure']: os.getenv('FOUNDRY_IQ_PROCEDURES_INDEX') or os.getenv('AZURE_SEARCH_INDEX'),
    source_by_role['authorization']: os.getenv('FOUNDRY_IQ_RAAMOPDRACHTEN_INDEX') or os.getenv('AZURE_SEARCH_RO_INDEX'),
    source_by_role['crew']: os.getenv('FOUNDRY_IQ_CREW_INDEX') or os.getenv('AZURE_SEARCH_CREW_INDEX'),
}
if any(not source or not index for source, index in source_to_index.items()):
    raise ValueError('All three Foundry IQ source and index mappings are required')
procedure_threshold = float(os.getenv('FOUNDRY_IQ_PROCEDURE_RERANKER_THRESHOLD', '2.0'))
structured_threshold = float(os.getenv('FOUNDRY_IQ_STRUCTURED_RERANKER_THRESHOLD', '1.5'))
source_candidate_cap = int(os.getenv('FOUNDRY_IQ_SOURCE_CANDIDATE_CAP', '50'))
if not 50 <= source_candidate_cap <= 200:
    raise ValueError('FOUNDRY_IQ_SOURCE_CANDIDATE_CAP must be between 50 and 200')

def source_parameters_for(case: dict) -> list[dict]:
    return [
        {
            'knowledgeSourceName': source_by_role[role],
            'kind': 'searchIndex',
            'includeReferences': True,
            'includeReferenceSourceData': True,
            'alwaysQuerySource': True,
            'failOnError': True,
            'rerankerThreshold': procedure_threshold if role == 'procedure' else structured_threshold,
            'maxOutputDocuments': source_candidate_cap,
            'filterAddOn': case['source_filters'][role],
        }
        for role in ('procedure', 'authorization', 'crew')
    ]

clients_by_source = {
    source: SearchClient(endpoint=search_endpoint, index_name=index, credential=credential)
    for source, index in source_to_index.items()
}
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()
rates = price_rates_from_env()
retrieve_url = (
    f"{search_endpoint}/knowledgebases/{quote(knowledge_base, safe='')}"
    f"/retrieve?api-version={api_version}"
)
print({'sources': source_to_index, 'source_candidate_cap': source_candidate_cap, 'retrieve_url': retrieve_url.split('?')[0]})


In [ ]:
def retrieve_from_iq(case: dict) -> tuple[int, dict]:
    client_request_id = str(__import__('uuid').uuid4())
    body = {
        'messages': [
            {'role': 'user', 'content': [{'type': 'text', 'text': case['query']}]}
        ],
        'knowledgeSourceParams': source_parameters_for(case),
        'includeActivity': True,
        'outputMode': iq_output_mode,
        'maxOutputDocuments': 12,
        'maxOutputSize': 20000,
    }
    request = urllib.request.Request(
        retrieve_url,
        data=json.dumps(body).encode('utf-8'),
        method='POST',
        headers={
            'Authorization': f'Bearer {search_token}',
            'Content-Type': 'application/json',
            'x-ms-client-request-id': client_request_id,
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=240) as response:
            payload = json.load(response)
            payload['_client_request_id'] = client_request_id
            return response.status, payload
    except urllib.error.HTTPError as error:
        detail = error.read().decode('utf-8', errors='replace')
        raise RuntimeError(
            f'Foundry IQ retrieve failed with HTTP {error.code}; '
            f'client_request_id={client_request_id}: {detail[:2000]}'
        ) from error


def generate_cited_answer(query: str, context: str, valid_reference_ids: set[str]):
    ordered_ids = sorted(valid_reference_ids, key=lambda value: int(value.split(':', 1)[1]))
    schema = {
        'type': 'object',
        'properties': {
            'claims': {
                'type': 'array',
                'items': {
                    'type': 'object',
                    'properties': {
                        'text': {'type': 'string'},
                        'source_ids': {
                            'type': 'array',
                            'items': {'type': 'string', 'enum': ordered_ids},
                        },
                    },
                    'required': ['text', 'source_ids'],
                    'additionalProperties': False,
                },
            },
            'insufficient_evidence': {
                'type': 'array',
                'items': {'type': 'string'},
            },
        },
        'required': ['claims', 'insufficient_evidence'],
        'additionalProperties': False,
    }
    response = openai_client.responses.create(
        model=model_deployment,
        instructions=(
            'Answer only from the supplied Foundry IQ evidence. Return one independent, '
            'single-sentence factual claim per claims item and list every reference ID that '
            'supports that claim. Cover the requested procedure, authorization scope and '
            'validity, and crew linkage and availability. Do not combine unrelated facts, '
            'invent facts, or use a reference ID outside the allowed enum. If evidence for '
            'a requested point is absent, put that point in insufficient_evidence instead.'
        ),
        input=f"Question: {query}\n\nFoundry IQ evidence:\n{context}",
        text={
            'format': {
                'type': 'json_schema',
                'name': 'cited_rag_answer',
                'strict': True,
                'schema': schema,
            }
        },
        max_output_tokens=1000,
    )
    validation = validate_cited_answer(json.loads(response.output_text), valid_reference_ids)
    if validation['valid_coverage'] != 1.0 or validation['validity'] != 1.0:
        raise ValueError(f'Generated answer failed deterministic citation validation: {validation}')
    return response, validation, render_cited_answer(validation)

print('Foundry IQ retrieval and claim-level answer helpers ready.')

## Execute the repeated live IQ → cited-answer path

The expected answer is never supplied to generation. It is used only later by the completeness evaluator. HTTP 206, source warnings, missing evidence, and invalid citations remain explicit release failures.

In [ ]:
iq_rows = []
for case in iq_cases:
    for repeat_number in range(1, repeat_count + 1):
        run_case_id = f"{case['case_id']}-R{repeat_number}"
        total_started = time.perf_counter()
        retrieval_started = time.perf_counter()
        status_code, payload = retrieve_from_iq(case)
        retrieve_ms = (time.perf_counter() - retrieval_started) * 1000
        extracted_documents = extract_iq_extracted_documents(payload)
        resolved = fetch_iq_reference_documents(payload, clients_by_source)
        if not extracted_documents or not resolved:
            raise RuntimeError(f'{run_case_id}: Foundry IQ returned no extracted evidence')

        context = format_iq_context(resolved)
        retrieved_keys = set().union(
            *(document_evidence_keys(reference['document']) for reference in resolved)
        )
        recall = retrieval_recall(case['expected_evidence_groups'], retrieved_keys)
        valid_reference_ids = {f"ref_id:{reference['ref_id']}" for reference in resolved}

        generation_started = time.perf_counter()
        response, cited_value, answer = generate_cited_answer(case['query'], context, valid_reference_ids)
        generation_ms = (time.perf_counter() - generation_started) * 1000
        citations = cited_value
        activity = iq_activity_metrics(payload.get('activity') or [])
        generation_usage = response_token_usage(response, semantic_requests=0)
        for metric in ('model_input_tokens', 'model_output_tokens', 'agentic_retrieval_tokens', 'semantic_requests'):
            activity[metric] += generation_usage[metric]
        cost = estimate_cost_usd(activity, rates)
        source_warnings = [
            {
                'source': item.get('knowledgeSourceName'),
                'warning': item.get('warning'),
                'error': item.get('error'),
            }
            for item in payload.get('activity') or []
            if item.get('warning') or item.get('error')
        ]
        row = {
            'case_id': run_case_id,
            'benchmark_case_id': case['case_id'],
            'repeat': repeat_number,
            'query': case['query'],
            'ground_truth': case['ground_truth'],
            'context': context,
            'response': answer,
            'http_status': status_code,
            'client_request_id': payload['_client_request_id'],
            'references': len(resolved),
            'serialized_documents': len(extracted_documents),
            'retrieval_recall_at_12': recall['recall'],
            'recall_details': recall['groups'],
            'citation_coverage': citations['coverage'],
            'citation_validity': citations['validity'],
            'invalid_citations': citations['invalid_citations'],
            'insufficient_evidence': cited_value['insufficient_evidence'],
            'retrieve_ms': round(retrieve_ms, 2),
            'generation_ms': round(generation_ms, 2),
            'total_ms': round((time.perf_counter() - total_started) * 1000, 2),
            'source_warnings': source_warnings,
            **activity,
            **cost,
        }
        iq_rows.append(row)
        print(json.dumps({key: row[key] for key in ('case_id', 'client_request_id', 'http_status', 'references', 'serialized_documents', 'retrieval_recall_at_12', 'recall_details', 'citation_coverage', 'citation_validity', 'insufficient_evidence', 'retrieve_ms', 'generation_ms', 'total_ms', 'model_input_tokens', 'model_output_tokens', 'agentic_retrieval_tokens', 'semantic_requests', 'estimated_cost_usd', 'source_warnings')}, indent=2))

assert len(iq_rows) == len(iq_cases) * repeat_count
assert all(row['response'] and row['context'] for row in iq_rows)
print('PASS — every versioned case completed every live repeat with extractive IQ evidence and a separately generated cited answer.')

## Evaluate groundedness and response completeness

Groundedness checks whether the answer is supported by the exact IQ references. Response completeness checks whether the supported answer covers the independently versioned expected facts. Each frozen response is judged three times by default; raw samples and the mean are retained, and a predeclared median is the consensus gate. Evaluator token use is reported separately from application-path cost.

In [ ]:
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom

data_source_config = DataSourceConfigCustom(
    type='custom',
    item_schema={
        'type': 'object',
        'properties': {
            'eval_item_id': {'type': 'string'},
            'run_case_id': {'type': 'string'},
            'judge_repeat': {'type': 'integer'},
            'query': {'type': 'string'},
            'context': {'type': 'string'},
            'response': {'type': 'string'},
            'ground_truth': {'type': 'string'},
        },
        'required': ['eval_item_id', 'run_case_id', 'judge_repeat', 'query', 'context', 'response', 'ground_truth'],
    },
)
groundedness_criterion = TestingCriterionAzureAIEvaluator(
    type='azure_ai_evaluator',
    name='groundedness',
    evaluator_name='builtin.groundedness',
    initialization_parameters={'deployment_name': evaluator_model},
    data_mapping={
        'query': '{{item.query}}',
        'context': '{{item.context}}',
        'response': '{{item.response}}',
    },
)
completeness_criterion = TestingCriterionAzureAIEvaluator(
    type='azure_ai_evaluator',
    name='response_completeness',
    evaluator_name='builtin.response_completeness',
    initialization_parameters={'deployment_name': evaluator_model},
    data_mapping={
        'ground_truth': '{{item.ground_truth}}',
        'response': '{{item.response}}',
    },
)
run_suffix = str(int(time.time()))
eval_object = openai_client.evals.create(
    name=f'd2-e2e-iq-{resource_namespace}-{run_suffix}',
    data_source_config=data_source_config,
    testing_criteria=[groundedness_criterion, completeness_criterion],
)
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name=f'd2-e2e-iq-run-{resource_namespace}-{run_suffix}',
    metadata={'namespace': resource_namespace, 'dataset': dataset['dataset_id'], 'target': 'foundry-iq-extractive-data', 'judge_repeats': str(judge_repeat_count), 'evaluator_model': evaluator_model},
    data_source={
        'type': 'jsonl',
        'source': {
            'type': 'file_content',
            'content': [
                {
                    'item': {
                        'eval_item_id': f"{row['case_id']}-J{judge_repeat}",
                        'run_case_id': row['case_id'],
                        'judge_repeat': judge_repeat,
                        **{key: row[key] for key in ('query', 'context', 'response', 'ground_truth')},
                    }
                }
                for row in iq_rows
                for judge_repeat in range(1, judge_repeat_count + 1)
            ],
        },
    },
)
print({'evaluation_id': eval_object.id, 'run_id': eval_run.id})

In [ ]:
deadline = time.monotonic() + 20 * 60
while eval_run.status not in ('completed', 'failed', 'canceled'):
    if time.monotonic() > deadline:
        raise TimeoutError('RAG evaluation exceeded 20 minutes')
    time.sleep(5)
    eval_run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    print('status:', eval_run.status)
output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
if eval_run.status != 'completed':
    run_error = getattr(eval_run, 'error', None)
    if hasattr(run_error, 'model_dump'):
        run_error = run_error.model_dump(mode='json')
    raise RuntimeError(f'Evaluation infrastructure failure: {{"status": {eval_run.status!r}, "eval_id": {eval_object.id!r}, "run_id": {eval_run.id!r}, "server_error": {run_error!r}, "output_items": {len(output_items)}, "report_url": {getattr(eval_run, "report_url", None)!r}}}')
expected_output_items = len(iq_rows) * judge_repeat_count
output_deadline = time.monotonic() + 2 * 60
while len(output_items) < expected_output_items:
    if time.monotonic() > output_deadline:
        raise TimeoutError(f'Evaluation completed but exposed only {len(output_items)}/{expected_output_items} output items')
    time.sleep(2)
    output_items = list(openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id))
assert len(output_items) == expected_output_items
metrics = ('groundedness', 'response_completeness')
scores_by_case = {
    row['case_id']: {metric: {} for metric in metrics}
    for row in iq_rows
}
for item in output_items:
    data = primitive(item)
    source_item = data['datasource_item']
    case_id = source_item['run_case_id']
    judge_repeat = int(source_item['judge_repeat'])
    for metric in metrics:
        result = next(result for result in data['results'] if result['name'] == metric)
        if result.get('error') or result.get('status') in ('failed', 'error', 'canceled'):
            raise RuntimeError(f'{case_id}-J{judge_repeat}: {metric} evaluator failed: {result}')
        if judge_repeat in scores_by_case[case_id][metric]:
            raise RuntimeError(f'{case_id}: duplicate {metric} judge repeat {judge_repeat}')
        scores_by_case[case_id][metric][judge_repeat] = float(result['score'])
for row in iq_rows:
    for metric in metrics:
        samples = [
            scores_by_case[row['case_id']][metric][repeat]
            for repeat in range(1, judge_repeat_count + 1)
        ]
        row[f'{metric}_samples'] = samples
        row[f'{metric}_mean'] = round(sum(samples) / len(samples), 3)
        row[metric] = float(median(samples))

evaluator_usage = [primitive(item) for item in (getattr(eval_run, 'per_model_usage', None) or [])]
print({
    'judge_consensus': {
        row['case_id']: {
            metric: {
                'samples': row[f'{metric}_samples'],
                'mean': row[f'{metric}_mean'],
                'median': row[metric],
            }
            for metric in metrics
        }
        for row in iq_rows
    },
    'evaluator_model': evaluator_model,
    'evaluator_usage': evaluator_usage,
    'report_url': getattr(eval_run, 'report_url', None),
})
assert len(scores_by_case) == len(iq_rows)
print('PASS — Foundry returned three judge samples and median groundedness/completeness consensus for every live run.')

## Release view and success check

The gates are intentionally strict: every expected evidence group must be retrieved, every factual claim must be cited with a valid returned reference, no requested evidence may be missing, both judge scores must be at least 4/5, and the retrieve operation must be complete and warning-free.

In [ ]:
thresholds = {
    'retrieval_recall_at_12': 1.0,
    'citation_coverage': 1.0,
    'citation_validity': 1.0,
    'groundedness': 4.0,
    'response_completeness': 4.0,
}
for row in iq_rows:
    deterministic_ok = all(
        row[metric] >= thresholds[metric]
        for metric in ('retrieval_recall_at_12', 'citation_coverage', 'citation_validity')
    )
    judge_ok = all(
        row[metric] >= thresholds[metric]
        for metric in ('groundedness', 'response_completeness')
    )
    operational_ok = row['http_status'] == 200 and not row['source_warnings']
    evidence_ok = not row['insufficient_evidence'] and row['serialized_documents'] > 0
    row['deterministic_gate_passed'] = deterministic_ok and operational_ok and evidence_ok
    row['groundedness_pass_rate'] = round(
        sum(score >= thresholds['groundedness'] for score in row['groundedness_samples'])
        / len(row['groundedness_samples']),
        3,
    )
    row['response_completeness_pass_rate'] = round(
        sum(score >= thresholds['response_completeness'] for score in row['response_completeness_samples'])
        / len(row['response_completeness_samples']),
        3,
    )
    row['release_gate_passed'] = row['deterministic_gate_passed'] and judge_ok
    print(json.dumps({
        'case_id': row['case_id'],
        'recall@12': row['retrieval_recall_at_12'],
        'citation_coverage': row['citation_coverage'],
        'citation_validity': row['citation_validity'],
        'groundedness': {
            'samples': row['groundedness_samples'],
            'mean': row['groundedness_mean'],
            'median': row['groundedness'],
            'pass_rate': row['groundedness_pass_rate'],
        },
        'response_completeness': {
            'samples': row['response_completeness_samples'],
            'mean': row['response_completeness_mean'],
            'median': row['response_completeness'],
            'pass_rate': row['response_completeness_pass_rate'],
        },
        'latency_ms': {
            'iq_retrieval': row['retrieve_ms'],
            'answer_generation': row['generation_ms'],
            'total': row['total_ms'],
            'query_planning': row['query_planning_ms'],
            'search_elapsed_sum': row['search_execution_ms_sum'],
        },
        'usage': {
            'model_input_tokens': row['model_input_tokens'],
            'model_output_tokens': row['model_output_tokens'],
            'agentic_retrieval_tokens': row['agentic_retrieval_tokens'],
            'search_requests': row['semantic_requests'],
        },
        'estimated_cost_usd': row['estimated_cost_usd'],
        'release_gate_passed': row['release_gate_passed'],
    }, indent=2))

case_summaries = {}
for case in iq_cases:
    rows = [row for row in iq_rows if row['benchmark_case_id'] == case['case_id']]
    ordered_latency = sorted(row['total_ms'] for row in rows)
    p95_index = min(len(ordered_latency) - 1, math.ceil(0.95 * len(ordered_latency)) - 1)
    case_summaries[case['case_id']] = {
        'runs': len(rows),
        'min_recall_at_12': min(row['retrieval_recall_at_12'] for row in rows),
        'min_citation_coverage': min(row['citation_coverage'] for row in rows),
        'min_citation_validity': min(row['citation_validity'] for row in rows),
        'min_groundedness_median': min(row['groundedness'] for row in rows),
        'min_groundedness_mean': min(row['groundedness_mean'] for row in rows),
        'min_response_completeness_median': min(row['response_completeness'] for row in rows),
        'min_response_completeness_mean': min(row['response_completeness_mean'] for row in rows),
        'judge_disagreement_runs': sum(
            len(set(row['groundedness_samples'])) > 1
            or len(set(row['response_completeness_samples'])) > 1
            for row in rows
        ),
        'p95_total_ms': ordered_latency[p95_index],
        'all_release_gates_passed': all(row['release_gate_passed'] for row in rows),
    }
print(json.dumps({'case_summaries': case_summaries}, indent=2))

assert all(row['release_gate_passed'] for row in iq_rows), 'One or more live IQ release gates failed; inspect the row-level diagnostics above.'
print('SUCCESS CHECK — every repeated live IQ run passed retrieval, citation, groundedness, completeness, operational, latency and cost checks.')

## Participant challenge

Add a multi-source case that should return a safe `insufficient_evidence` result. Define its expected evidence groups and source filters first, then create a separate expected-failure gate instead of weakening the release thresholds above.

In [ ]:
# TODO: add a synthetic missing-evidence case with its own expected-failure assertion.
# Keep retrieval, generation, citation, completeness, and operational failures separate.

In [ ]:
allow_cleanup = os.getenv('WORKSHOP_ALLOW_CLEANUP', 'false').lower() == 'true'
if allow_cleanup:
    openai_client.evals.delete(eval_id=eval_object.id)
    print('Deleted this namespaced evaluation and its run.')
else:
    print('Cleanup disabled so the Foundry evaluation report remains available.')